In [1]:
import cv2 as cv
import threading
import time
import ipywidgets as widgets
from IPython.display import display
from gesture_stacking import Gesture_Stacking
from dofbot_utils.fps import FPS
from dofbot_utils.robot_controller import Robot_Controller

In [2]:
robot = Robot_Controller()
robot.move_init_pose()
fps = FPS()

gesture = Gesture_Stacking()
model = 'General'

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [3]:
button_layout      = widgets.Layout(width='320px', height='60px', align_self='auto')
output = widgets.Output()

# 退出 exit
exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)

# 图像控件 Image widget
imgbox = widgets.Image(format='jpg', height=480, width=640)
# 垂直布局 Vertical layout
display_box = widgets.VBox([imgbox, exit_button])

In [4]:
def exit_button_Callback(value):
    global model
    model = 'Exit'
    with output:
        print(model)

exit_button.on_click(exit_button_Callback)

In [ ]:
def camera():
    global model, gesture
    # 打开摄像头 Open camera
    capture = cv.VideoCapture(0, cv.CAP_V4L2)
    capture.set(3, 640)
    capture.set(4, 480)
    # Be executed in loop when the camera is opened normally 
    # 当摄像头正常打开的情况下循环执行
    while capture.isOpened():
        try:
            _, img = capture.read()
            fps.update_fps()
            gesture.process(img)
            if model == 'Exit':
                capture.release()
                del gesture
                break
            fps.show_fps(img)
            imgbox.value = cv.imencode('.jpg', img)[1].tobytes()
            time.sleep(0.002)
        except Exception as e:
            print("program end")
            print(e)
            capture.release()

In [6]:
display(display_box,output)
threading.Thread(target=camera, ).start()

Output()